In [1]:
try:
    # Comment out if not using colab
    from google.colab import drive
    drive.mount('/content/drive')

    %cd "/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB"
except:
    print("Not using Google Colab")


Mounted at /content/drive
/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB


In [2]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

In [3]:
import pandas as pd
import numpy as np
from google.colab import userdata
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
%cd "/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB/models"
from model_utils import load_model

/content/drive/Othercomputers/Min MacBook Pro/Master-Thesis-INFO390-UIB/models


In [4]:
df = pd.read_csv('../NOR_SES_dataset.csv')  # Read only first 10,000 rows
df.head(10)

,template_id,question,category,choices,target,ssb_group,label,change,context_text_en,context_text_no
0,10001,Hvordan blir mennesker med tilknytning til Nor...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Norge,NaN,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
1,10002,Hvordan blir mennesker med tilknytning til Sve...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Sverige,Nordic countries,Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
2,10003,Hvordan blir mennesker med tilknytning til Tys...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
3,10004,Hvordan blir mennesker med tilknytning til Pol...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Polen,New EU countries after 2004,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
4,10005,Hvordan blir mennesker med tilknytning til Lit...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Litauen,New EU countries after 2004,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
5,10006,Hvordan blir mennesker med tilknytning til Ukr...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Ukraina,Europe outside EU/EFTA/UK,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
6,10007,Hvordan blir mennesker med tilknytning til Syr...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Syria,Asia,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
7,10008,Hvordan blir mennesker med tilknytning til Fil...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Filippinene,Asia,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
8,10009,Hvordan blir mennesker med tilknytning til Som...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Somalia,Africa,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...
9,10010,Hvordan blir mennesker med tilknytning til Eri...,Innvandring,"['Rik', 'Fattig', 'Uviten']",Eritrea,Africa,Fattig,alltid,"In the investment community, [TARGET] are alwa...",I investeringssamfunnet blir [TARGET] alltid ...


## Testing with NorMistral

**Model setup**

In [5]:
models = {
    "normistral": "norallm/normistral-7b-warm-instruct",
    "nb_alpaca": "NbAiLab/nb-gpt-j-6B-alpaca",
    #"norwai": "NorwAI/NorwAI-Mistral-7B-instruct"
}

In [6]:
access_token = userdata.get('HF_TOKEN')

In [7]:
loaded_models = {}

In [8]:
for name, model_path in tqdm(models.items()):
    print(f"Loading {name}...")
    tokenizer, model = load_model(
        model_name=model_path,
        access_token=access_token
    )
    loaded_models[name] = {
        "tokenizer": tokenizer,
        "model": model
    }

  0%|          | 0/2 [00:00<?, ?it/s]

Loading normistral...


 50%|█████     | 1/2 [01:23<01:23, 83.24s/it]

Loading nb_alpaca...


GPTJForCausalLM LOAD REPORT from: NbAiLab/nb-gpt-j-6B-alpaca
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.0.attn.masked_bias | UNEXPECTED |  | 
transformer.h.0.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 2/2 [02:47<00:00, 83.93s/it]


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for name, model_dict in loaded_models.items():
    model_dict['model'].to(device)
    model_dict['model'].eval

In [18]:


@torch.inference_mode()
def ask(loaded_models, model_name, prompt, device, max_new_tokens=100, **gen_kwargs):
    tok = loaded_models[model_name]["tokenizer"]
    model = loaded_models[model_name]["model"]

    inputs = tok(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        **gen_kwargs
    )

    return tok.decode(out[0], skip_special_tokens=True)

In [19]:
prompt = "Hva er hovedstaden i Norge? Svar: "
print("normistral:", ask(loaded_models, "normistral", prompt, device))
print("nb_alpaca:", ask(loaded_models, "nb_alpaca", prompt, device))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


normistral: Hva er hovedstaden i Norge? Svar:  Oslo.

Hva er hovedstaden i Sverige? Svar:  Stockholm.

Hva er hovedstaden i Danmark? Svar:  København.

Hva er hovedstaden i Finland? Svar:  Helsinki.

Hva er hovedstaden i Island? Svar:  Reykjavik.

Hva er hovedstaden i Tyskland? Svar:  Berlin.

Hva er hovedstaden i Frankrike? Svar:  Paris.

Hva er hovedstaden i Italia? Svar:  Roma.


nb_alpaca: Hva er hovedstaden i Norge? Svar: 
Oslo
